# Kaggle S6E4 — Public OOF Stacking (TunedBlend)
**Runtime: ~2 min** | Loads OOF predictions from public notebooks and blends them.

Inspired by [ravi20076/PlaygroundS6E4|TunedBlend|V1](https://www.kaggle.com/code/ravi20076) (LB 0.98018)

In [ ]:
import numpy as np
import pandas as pd
import os, glob
from itertools import permutations
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from scipy.optimize import minimize

print('Libraries loaded')

## 1. Load Competition Data

In [ ]:
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e4'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
sub   = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

TARGET = 'Irrigation_Need'
le = LabelEncoder()
y = le.fit_transform(train[TARGET])
n_train, n_test = len(train), len(test)
print(f'Train: {n_train}, Test: {n_test}')
print(f'Classes: {dict(zip(le.classes_, le.transform(le.classes_)))}')

## 2. Load Public OOF Predictions

**Required Kaggle Inputs** (Add Input → Notebook Output / Dataset):
- `yunsuxiaozi/pss6e4-xgb-cv-0-979805`
- `yunsuxiaozi/pss6e4-lgb-advanced-cv-0-97997`
- `yunsuxiaozi/pss6e4-lgb-baselinecv-0-97943`
- `utaazu/0-979-cv-single-cat-pairwise-te-bias-tuning`
- `mahoganybuttstrings/pg-s6e4-realmlp-cv-0-97802-lb-0-97685`
- `ravi20076/playgrounds6e4publicmodelsv1` (dataset)
- `ravi20076/playgrounds6e4-public-baseline-v3` (notebook)
- `wguesdon/ps6e4-irrigation-14-model-predictions` (dataset)

In [ ]:
# Show all available input files
for dirname, _, filenames in os.walk('/kaggle/input'):
    for f in filenames:
        path = os.path.join(dirname, f)
        if f.endswith(('.npy', '.parquet', '.csv')) and 'playground-series' not in path:
            print(path)

In [ ]:
def find_best_perm(oof, y_true):
    """Try all 6 column permutations, return the one with best balanced acc."""
    best_perm, best_score = (0, 1, 2), -1
    for perm in permutations(range(3)):
        s = balanced_accuracy_score(y_true, np.argmax(oof[:, list(perm)], axis=1))
        if s > best_score:
            best_score = s
            best_perm = perm
    return list(best_perm), best_score


def load_all_oofs():
    """Auto-detect all OOF/test prediction pairs under /kaggle/input."""
    base = '/kaggle/input'
    found = []

    # Known patterns: (name, oof_glob, test_glob, format)
    patterns = [
        ('GNN_v4',    '**/OOF_Preds_GNNV4_1.parquet', '**/Mdl_Preds_GNNV4_1.parquet', 'parquet'),
        ('FTT_v4',    '**/OOF_Preds_FTTV4_1.parquet', '**/Mdl_Preds_FTTV4_1.parquet', 'parquet'),
        ('Baseline',  '**/playgrounds6e4-public-baseline*/OOF_Preds.npy',
                      '**/playgrounds6e4-public-baseline*/Mdl_Preds.npy', 'npy'),
        ('RealMLP',   '**/realmlp_oof.csv', '**/realmlp_pred.csv', 'csv3'),
        ('Cat_pair',  '**/oof_probs_cat.npy', '**/test_probs_cat.npy', 'npy'),
        ('LGB_base',  '**/pss6e4-lgb-baselinecv*/oof_preds.npy',
                      '**/pss6e4-lgb-baselinecv*/test_preds.npy', 'npy'),
        ('LGB_adv',   '**/pss6e4-lgb-advanced*/oof_preds.npy',
                      '**/pss6e4-lgb-advanced*/test_preds.npy', 'npy'),
        ('XGB_OTE',   '**/pss6e4-xgb-cv*/oof_preds.npy',
                      '**/pss6e4-xgb-cv*/test_preds.npy', 'npy'),
    ]

    for name, oof_pat, test_pat, fmt in patterns:
        oof_paths  = glob.glob(os.path.join(base, oof_pat),  recursive=True)
        test_paths = glob.glob(os.path.join(base, test_pat), recursive=True)
        if not oof_paths or not test_paths:
            continue
        try:
            if fmt == 'parquet':
                oof = pd.read_parquet(oof_paths[0]).values
                tp  = pd.read_parquet(test_paths[0]).values
            elif fmt == 'npy':
                oof = np.load(oof_paths[0])
                tp  = np.load(test_paths[0])
            elif fmt == 'csv3':
                oof = pd.read_csv(oof_paths[0]).values.reshape(n_train, 3)
                tp  = pd.read_csv(test_paths[0]).values.reshape(n_test, 3)
            else:
                continue
            if oof.shape != (n_train, 3) or tp.shape != (n_test, 3):
                print(f'  [skip] {name}: shape mismatch {oof.shape}/{tp.shape}')
                continue
            perm, score = find_best_perm(oof, y)
            oof, tp = oof[:, perm], tp[:, perm]
            print(f'  [ok] {name}: perm={perm}, score={score:.5f}')
            found.append((name, oof, tp, score))
        except Exception as e:
            print(f'  [err] {name}: {e}')

    # 14-model dataset
    for p in glob.glob(os.path.join(base, '**/ps6e4-irrigation*/oof_*.npy'), recursive=True):
        try:
            tp_path = p.replace('oof_', 'pred_')
            if not os.path.exists(tp_path): continue
            oof = np.load(p)
            tp  = np.load(tp_path)
            if oof.shape != (n_train, 3) or tp.shape != (n_test, 3): continue
            perm, score = find_best_perm(oof, y)
            oof, tp = oof[:, perm], tp[:, perm]
            name = 'wgu_' + os.path.basename(p).replace('oof_','').replace('.npy','')
            print(f'  [ok] {name}: perm={perm}, score={score:.5f}')
            found.append((name, oof, tp, score))
        except Exception as e:
            print(f'  [err] {e}')

    return found


print('Scanning for OOFs...\n')
all_oofs = load_all_oofs()
print(f'\nTotal: {len(all_oofs)} models found')

## 3. TunedBlender — Per-Fold Per-Class Scaling

In [ ]:
if len(all_oofs) == 0:
    raise RuntimeError('No OOFs found! Add notebook outputs as inputs first.')

# Filter: keep models within 0.008 of best
best_score = max(s for _, _, _, s in all_oofs)
cutoff = best_score - 0.008
kept = [(n, o, t, s) for n, o, t, s in all_oofs if s >= cutoff]
print(f'Keeping {len(kept)}/{len(all_oofs)} models above cutoff {cutoff:.5f}:')
for n, _, _, s in sorted(kept, key=lambda x: -x[3]):
    print(f'  {n}: {s:.5f}')

# Stack horizontally
Xtr = np.hstack([o for _, o, _, _ in kept]).astype('float32')
Xte = np.hstack([t for _, _, t, _ in kept]).astype('float32')
n_models = len(kept)
print(f'\nStacked shapes: train={Xtr.shape}, test={Xte.shape}')

In [ ]:
# TunedBlender: per-fold tuned per-class scaling
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_blend = np.zeros((n_train, 3), dtype='float32')
test_per_fold = []
fold_scales = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(Xtr, y)):
    avg_tr  = Xtr[tr_idx].reshape(-1, n_models, 3).mean(axis=1)
    avg_val = Xtr[val_idx].reshape(-1, n_models, 3).mean(axis=1)
    avg_te  = Xte.reshape(-1, n_models, 3).mean(axis=1)
    y_tr    = y[tr_idx]

    def _neg(scale):
        s = np.clip(scale, 1e-3, None)
        return -balanced_accuracy_score(y_tr, np.argmax(avg_tr * s, axis=1))

    res = minimize(_neg, x0=np.ones(3), method='Nelder-Mead',
                   options={'xatol': 1e-4, 'fatol': 1e-6, 'maxiter': 2000})
    scale = np.clip(res.x, 1e-3, None)
    fold_scales.append(scale)

    oof_blend[val_idx] = avg_val * scale
    test_per_fold.append(avg_te * scale)
    print(f'  Fold {fold+1}: scale={scale.round(5)}, '
          f'val_bal_acc={balanced_accuracy_score(y[val_idx], np.argmax(avg_val * scale, axis=1)):.5f}')

test_blend = np.mean(test_per_fold, axis=0)

print(f'\nMean scale: {np.mean(fold_scales, axis=0).round(5)}')
final_score = balanced_accuracy_score(y, np.argmax(oof_blend, axis=1))
print(f'\nFinal OOF Balanced Accuracy: {final_score:.5f}')

## 4. Submission

In [ ]:
final_labels = le.inverse_transform(np.argmax(test_blend, axis=1))
sub[TARGET] = final_labels
sub.to_csv('submission.csv', index=False)

print('submission.csv saved!')
print(sub[TARGET].value_counts())
sub.head(10)

In [ ]:
# Save OOF predictions for potential re-use by other notebooks
np.save('oof_blend.npy', oof_blend)
np.save('test_blend.npy', test_blend)
print('OOF + test blend predictions saved as .npy')